In [1]:
print("Hello, World!")

Hello, World!


In [ ]:
# from ragas.metrics.collections import (
#     ContextPrecision,
#     ContextRecall,
#     ContextEntityRecall,
#     NoiseSensitivity,
#     AnswerRelevancy,
#     Faithfulness,
#     FactualCorrectness,
#     SemanticSimilarity,
# )
# from ragas.metrics.collections import (
#     MultiModalFaithfulness,
#     MultiModalRelevance,
# )
# from ragas.metrics.collections import (
#     TopicAdherence,
#     ToolCallAccuracy,
#     ToolCallF1,
#     AgentGoalAccuracyWithReference,
#     AgentGoalAccuracyWithoutReference,
# )
# from ragas.metrics.collections import (
#     FactualCorrectness,
#     SemanticSimilarity,
#     NonLLMStringSimilarity,
#     DistanceMeasure,
#     BleuScore,
#     CHRFScore,
#     RougeScore,
#     StringPresence,
#     ExactMatch,
# )

# from ragas.metrics import (
#     discrete_metric,
#     numeric_metric,
#     ranking_metric,
# )
# from ragas.metrics.collections import (
#     DomainSpecificRubrics,
#     InstanceSpecificRubrics,
# )

                             RAG
                              │
          ┌───────────────────┼───────────────────┐
          │                   │                   │
   OpenAIEmbeddings     Vector Store         Retriever
                                                  │
                                                  ▼
                                          Retrieved Docs
                                                  │
                                                  ▼
                                      ChatPromptTemplate
                                                  │
                                                  ▼
                                           ChatOpenAI
                                                  │
                                                  ▼
                                        StrOutputParser
                                                  │
                                                  ▼
                                             Response
                                                  │
                                                  │
                         RAGAS EVALUATION          │
                              │                   │
                              ▼                   ▼
                         Dataset              Response
                              │                   │
                              └─────────┬─────────┘
                                        ▼
                                   @experiment
                                        │
                         ┌──────────────┴─────────────┐
                         │                            │
                 DiscreteMetric                NumericMetric
                  Correctness                   Relevance
                         │                            │
                         └──────────────┬─────────────┘
                                        ▼
                                  Score + Reason
                                        │
                                        ▼
                                Experiment Results
                                        │
                                        ▼
                                      CSV

In [ ]:
# Rubric
# Likert

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env")

print("API key loaded successfully")

API key loaded successfully


In [36]:
MODEL = "gpt-4o"

EMBEDDING_MODEL = "text-embedding-3-small"

JUDGE_MODEL = "gpt-5.6-terra"

TOP_K = 2

In [4]:
from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings,
)

from langchain_core.documents import Document

from langchain_core.prompts import (
    ChatPromptTemplate,
)

from langchain_core.output_parsers import (
    StrOutputParser,
)

from langchain_core.vectorstores import (
    InMemoryVectorStore,
)

In [5]:
llm = ChatOpenAI(
    model=MODEL,
    temperature=0,
)

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
)

In [6]:
llm.invoke("hi how are you?")

AIMessage(content="Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 12, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_55c03d5f8a', 'id': 'chatcmpl-EkA2o5RzELskJyyZHqt0B9XFnQC3S', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1dab1-2628-7736-a37a-d97fc7ac4256-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 29, 'total_tokens': 41, 'input_token_details': {'audio':

In [30]:
# ============================================================
# 6. KNOWLEDGE BASE
# ============================================================

DOCUMENTS = [

    Document(
        page_content="""
Ragas is an open-source evaluation framework designed
for evaluating and improving LLM applications.

It supports RAG evaluation, agent evaluation,
custom metrics, datasets, experiments,
and synthetic test data generation.
""".strip(),
        metadata={
            "doc_id": "ragas_intro",
        },
    ),

    Document(
        page_content="""
Ragas experiments allow developers to systematically
evaluate AI applications, track evaluation results,
compare different versions, and store experiment outputs.
""".strip(),
        metadata={
            "doc_id": "ragas_experiments",
        },
    ),

    Document(
        page_content="""
Ragas metrics measure different properties of AI systems.

Metrics can evaluate factual correctness,
faithfulness, answer relevancy,
retrieval quality, agent behavior,
and application-specific requirements.
""".strip(),
        metadata={
            "doc_id": "ragas_metrics",
        },
    ),

    Document(
        page_content="""
RAG stands for Retrieval-Augmented Generation.

A RAG system retrieves relevant information
from a knowledge base and provides that information
to an LLM for answer generation.
""".strip(),
        metadata={
            "doc_id": "rag_intro",
        },
    ),

]

print(
    "\nKnowledge Base Documents:",
    len(DOCUMENTS),
)


Knowledge Base Documents: 4


In [31]:
vector_store = InMemoryVectorStore(embedding=embeddings,)

vector_store.add_documents(DOCUMENTS)

['00b90392-45de-4aa0-a98f-2b047c2a3eee',
 'eca20fb0-6edf-4f6c-99b1-360072f35207',
 '3778a989-d7c6-440e-b66e-a5531328a21f',
 'aba8c78e-07bf-4eea-8387-e1fc243e4762']

In [32]:
retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K,})

In [33]:
prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer the user's question ONLY using
the supplied context.

Rules:

1. Use only the provided context.
2. Do not invent information.
3. Do not use outside knowledge.
4. If the answer is not present in the context,
   say:

   "I don't know based on the provided context."

5. Keep the answer concise and factual.


CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""
)

In [34]:
def format_documents(documents,):
    return "\n\n".join(document.page_content for document in documents)

In [35]:
rag_chain = prompt | llm | StrOutputParser()

In [14]:
# ============================================================
# 12. COMPLETE LANGCHAIN RAG APPLICATION
# ============================================================

async def rag_query( question: str):
    
    # --------------------------------------------------------
    # STEP 1
    # Retrieve relevant documents using LangChain Retriever
    # --------------------------------------------------------

    relevant_documents = await retriever.ainvoke(question)
    
    retrieved_contexts = [ document.page_content for document in relevant_documents]
    
    retrieved_document_ids = [document.metadata.get("doc_id") for document in relevant_documents]

    # --------------------------------------------------------
    # STEP 2
    # Create context
    # --------------------------------------------------------

    context = format_documents(relevant_documents)

    # --------------------------------------------------------
    # STEP 3
    # Generate answer using LangChain LCEL chain
    # --------------------------------------------------------

    answer = await rag_chain.ainvoke(
        {
            "context": context,
            "question": question,
        }
    )

    return {

        "answer":answer,

        "retrieved_contexts": retrieved_contexts,

        "retrieved_document_ids": retrieved_document_ids

    }

In [15]:
import asyncio
from pathlib import Path
import pandas as pd

In [16]:
from ragas import (
    Dataset,
    experiment,
)

from ragas.llms import (
    llm_factory,
)

from ragas.metrics import (
    DiscreteMetric,
    NumericMetric,
)

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def load_dataset():
    Path("evals").mkdir(exist_ok=True)

    dataset = Dataset(
        name="ragas_langchain_dataset",
        backend="local/csv",
        root_dir="evals",
    )

    # golden dataset(ground truth)
    data_samples = [

        {
            "question": "What is Ragas?",

            "expected_answer": """
Ragas is an evaluation framework
for LLM applications.
""".strip(),
        },

        {
            "question": "How do Ragas experiments work?",

            "expected_answer": """
Ragas experiments provide systematic evaluation,
track results, compare different application versions,
and store experiment outputs.
""".strip(),
        },

        {
            "question": "What do Ragas metrics do?",

            "expected_answer": """
Ragas metrics evaluate properties and quality
of AI or LLM application outputs.
""".strip(),
        },

        {
            "question": "What is RAG?",

            "expected_answer": """
RAG stands for Retrieval-Augmented Generation.

It retrieves relevant information from a knowledge
source and provides that context to an LLM
before answer generation.
""".strip(),
        },

    ]

    for sample in data_samples:
        dataset.append(sample)

    dataset.save()

    return dataset

In [20]:
from openai import AsyncOpenAI

In [21]:
ragas_openai_client = AsyncOpenAI(
    api_key=os.environ[ "OPENAI_API_KEY"],
    timeout=60.0,
    max_retries=3,
)

In [37]:
evaluator_llm = llm_factory(
    JUDGE_MODEL,
    client=ragas_openai_client,
)

In [23]:
correctness_metric = DiscreteMetric(

    name="correctness",

    prompt="""
Evaluate whether the model response correctly answers
the user's question according to the expected answer.

QUESTION:

{question}


EXPECTED ANSWER:

{expected_answer}


MODEL RESPONSE:

{response}


Evaluation Rules:

PASS if:

1. The response contains the important information
   from the expected answer.

2. The response is factually correct.

3. The response answers the user's question.


FAIL if:

1. Important information is missing.

2. The response contradicts the expected answer.

3. The response does not answer the question.


Return only:

pass

or

fail
""",

    allowed_values=[
        "pass",
        "fail",
    ],

)

In [24]:
relevance_metric = NumericMetric(

    name="relevance",

    prompt="""
Evaluate how relevant the model response is
to the user's question.


QUESTION:

{question}


MODEL RESPONSE:

{response}


Use this rubric:

1 = Completely irrelevant

2 = Mostly irrelevant

3 = Partially relevant

4 = Mostly relevant

5 = Completely relevant


Return only the numerical score.
""",

    allowed_values=(
        1,
        5,
    ),

)

In [25]:
@experiment()
async def run_rag_evaluation(row):

    # --------------------------------------------------------
    # Get question
    # --------------------------------------------------------
    question = row["question"]

    # --------------------------------------------------------
    # Get expected answer
    # --------------------------------------------------------
    expected_answer = row["expected_answer"]


    # ========================================================
    # RUN LANGCHAIN RAG
    # ========================================================

    rag_result = await rag_query(question)

    response = rag_result["answer"]

    retrieved_contexts = rag_result["retrieved_contexts"]

    retrieved_document_ids = rag_result["retrieved_document_ids"]


    # ========================================================
    # RAGAS CORRECTNESS EVALUATION
    # ========================================================

    correctness_result = await correctness_metric.ascore(
        llm=evaluator_llm,
        question=question,
        expected_answer=expected_answer,
        response=response,
    )


    # ========================================================
    # RAGAS RELEVANCE EVALUATION
    # ========================================================

    relevance_result = await relevance_metric.ascore(
        llm=evaluator_llm,
        question=question,
        response=response,
    )


    # ========================================================
    # RETURN EXPERIMENT DATA
    # ========================================================

    return {
        **row,

        # Application Output
        "response": response,

        # Retrieval Information
        "retrieved_contexts": retrieved_contexts,
        "retrieved_document_ids": retrieved_document_ids,

        # Correctness
        "correctness": correctness_result.value,
        "correctness_reason": correctness_result.reason,

        # Relevance
        "relevance": relevance_result.value,
        "relevance_reason": relevance_result.reason,
    }

In [ ]:
async def main():

    print("\n" + "=" * 100)
    print("RAGAS + LANGCHAIN EVALUATION")
    print("=" * 100)

    # ========================================================
    # LOAD DATASET
    # ========================================================

    dataset = load_dataset()

    print("\nDataset loaded.")

    # ========================================================
    # RUN RAGAS EXPERIMENT
    # ========================================================

    results = await run_rag_evaluation.arun(
        dataset,
        name="ragas_langchain_experiment",
    )

    # ========================================================
    # CONVERT RESULTS TO PANDAS
    # ========================================================

    df = results.to_pandas()

    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED RESULTS")
    print("=" * 100)

    columns_to_show = [
        "question",
        "response",
        "correctness",
        "relevance",
    ]

    print(df[columns_to_show].to_string(index=False))

    # ========================================================
    # CORRECTNESS PASS RATE
    # ========================================================

    correctness_pass_rate = (df["correctness"].astype(str).str.lower().eq("pass").mean())

    # ========================================================
    # AVERAGE RELEVANCE
    # ========================================================

    average_relevance = pd.to_numeric(df["relevance"],errors="coerce",).mean()

    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)

    print(f"Correctness Pass Rate : {correctness_pass_rate:.1%}")
    print(f"Average Relevance     : {average_relevance:.2f}/5")

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    experiment_directory = Path("evals/experiments")
    experiment_directory.mkdir(parents=True, exist_ok=True)

    output_file = experiment_directory / "ragas_langchain_results.csv"

    df.to_csv(output_file, index=False)

    print("\nResults saved to:")
    print(output_file)

    # ========================================================
    # PRINT DETAILED REASONS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED EVALUATION REASONS")
    print("=" * 100)

    for _, result_row in df.iterrows():

        print("\nQUESTION:")
        print(result_row["question"])

        print("\nRESPONSE:")
        print(result_row["response"])

        print("\nCORRECTNESS:")
        print(result_row["correctness"])

        print("\nCORRECTNESS REASON:")
        print(result_row["correctness_reason"])

        print("\nRELEVANCE:")
        print(result_row["relevance"])

        print("\nRELEVANCE REASON:")
        print(result_row["relevance_reason"])

        print("\nRETRIEVED DOCUMENT IDS:")
        print(result_row["retrieved_document_ids"])

        print("\n" + "-" * 100)

In [ ]:
try:
    asyncio.run(main())

except RuntimeError as error:

    # Jupyter Notebook fallback
    if "asyncio.run() cannot be called" in str(error) or "event loop" in str(error).lower():
        import nest_asyncio
        nest_asyncio.apply()

        loop = asyncio.get_event_loop()
        loop.run_until_complete(main())

    else:
        raise


RAGAS + LANGCHAIN EVALUATION

Dataset loaded.


Running experiment: 100%|██████████| 4/4 [00:19<00:00,  4.81s/it]


DETAILED RESULTS
                      question                                                                                                                                                                                           response correctness  relevance
                What is Ragas?                                                                                               Ragas is an open-source evaluation framework designed for evaluating and improving LLM applications.        pass        5.0
     What do Ragas metrics do? Ragas metrics measure different properties of AI systems, including factual correctness, faithfulness, answer relevancy, retrieval quality, agent behavior, and application-specific requirements.        pass        5.0
                  What is RAG?             RAG stands for Retrieval-Augmented Generation. It is a system that retrieves relevant information from a knowledge base and provides that information to an LLM for answer generation.        pa

: 